In [2]:
import sys; sys.path.append("..")
import pandas as pd
from collections import Counter
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from src.data import load_data
from src.config import RANDOM_STATE
from src.experiment_log import ExperimentLogger

X_train, _, y_train, _, labels = load_data()
lg = ExperimentLogger("thai")

def under_cap(y, cap=5000):          # giam lop lon xuong toi da 5000 mau
    return {k: cap for k, v in Counter(y).items() if v > cap}

def smote_to(y, target=2000):        # nang lop hiem len 2000 mau
    return {k: target for k, v in Counter(y).items() if v < target}

def rf_factory():
    return RandomForestClassifier(n_estimators=100, max_depth=10, max_features="log2",
                                  n_jobs=-1, random_state=RANDOM_STATE)

def hgb_factory(**kw):
    return HistGradientBoostingClassifier(early_stopping=False,
                                          random_state=RANDOM_STATE, **kw)   # thay bang tham so hgb_tuned neu co

def make_variants(factory):
    us = lambda: RandomUnderSampler(sampling_strategy=under_cap, random_state=RANDOM_STATE)
    sm = lambda: SMOTE(sampling_strategy=smote_to, k_neighbors=3, random_state=RANDOM_STATE)
    return {
        "N2_under":       ImbPipeline([("us", us()), ("clf", factory())]),
        "N3_smote":       ImbPipeline([("sc", StandardScaler()), ("sm", sm()), ("clf", factory())]),
        "N4_under_smote": ImbPipeline([("us", us()), ("sc", StandardScaler()), ("sm", sm()), ("clf", factory())]),
    }

# N0 va N1 cho RF: lay tu log cua B (rf_d10_cw_None, rf_d10_cw_balanced), khong chay lai
# N0 cho HGB: hgb_tuned (da co tu Buoc 4); N1 cho HGB:
lg.run(hgb_factory(class_weight="balanced"), X_train, y_train, "hgb_N1_balanced",
       strategy="class_weight=balanced", notes="N1 tren HGB")

for base, factory in [("rf", rf_factory), ("hgb", hgb_factory)]:
    for vname, pipe in make_variants(factory).items():
        lg.run(pipe, X_train, y_train, f"{base}_{vname}",
               strategy=f"{vname} (cap=5000 / SMOTE target=2000, k=3)",
               notes="thi nghiem mat can bang")

In [3]:
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.metrics import precision_recall_fscore_support

def per_class_report(model, X, y):
    cv = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)
    pred = cross_val_predict(model, X, y, cv=cv)
    p, r, f, _ = precision_recall_fscore_support(y, pred, labels=range(12), zero_division=0)
    return pd.DataFrame({"precision": p, "recall": r, "f1": f}, index=labels).round(3)

rep = pd.concat({
    "N0": per_class_report(rf_factory(), X_train, y_train),
    "N3_smote": per_class_report(make_variants(rf_factory)["N3_smote"], X_train, y_train),
}, axis=1)
rep.to_csv("../results/cv/imbalance_per_class_rf.csv")
rep

N0                N3_smote              
                           precision recall     f1 precision recall     f1
ARP_poisioning                 0.981  0.991  0.986     0.982  0.990  0.986
DDOS_Slowloris                 0.983  0.969  0.976     0.887  0.979  0.931
DOS_SYN_Hping                  1.000  1.000  1.000     1.000  1.000  1.000
MQTT_Publish                   1.000  0.998  0.999     1.000  0.998  0.999
Metasploit_Brute_Force_SSH     0.880  0.759  0.815     0.759  0.759  0.759
NMAP_FIN_SCAN                  0.947  0.818  0.878     0.720  0.818  0.766
NMAP_OS_DETECTION              1.000  1.000  1.000     1.000  1.000  1.000
NMAP_TCP_scan                  0.999  0.998  0.998     0.999  0.998  0.998
NMAP_UDP_SCAN                  0.994  0.986  0.990     0.994  0.963  0.978
NMAP_XMAS_TREE_SCAN            1.000  0.998  0.999     1.000  0.998  0.999
Thing_Speak                    0.988  0.986  0.987     0.988  0.985  0.986
Wipro_bulb                     0.963  0.897  0.929     0.933  0.960  0.946